In [1]:
from openai import AsyncOpenAI
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from enum import Enum
from langchain.chat_models import init_chat_model
from IPython.display import display

load_dotenv("/mnt/c/Users/jsh27/CSProj/academy-harness/agent-squared/.env")
monitor_model = init_chat_model("gpt-4o-mini")


In [2]:
class status(Enum):
    STUCK = 1
    PROGRESSING = 2
    COMPLETED = 3

In [3]:
class MonitorState(TypedDict):

    # last 5 summaries
    reasoning_sums: list[str]

    # Compact summary of everything before
    last_sum: str

    # Combined current summary
    curr_sum: str

    curr_analysis: str
    curr_state: str

    status_history: list[status]

In [4]:
class stream_buffer:

    def __init__(self, )->None:
        #(Order, Buffer_text)
        self.buffer: list[tuple[int, str]] = []
        self.counter:int = 0
        self.done: bool = False

    def append(self, append_token:str) -> None:
        self.counter += 1
        self.buffer.append((self.counter, append_token))

    def return_snapshot(self, snapshots_to_return:int) -> list[str]:
        
        last_snapshot:int = self.counter - snapshots_to_return
        returned_buffer = []

        for item in self.buffer:
            if item[0] >= last_snapshot:
                returned_buffer.append(item[1])

        return returned_buffer

In [5]:
# Nodes
async def compact_node(state: MonitorState):

    total_sum = "".join(state["reasoning_sums"])

    prompt = f"""You are monitoring an AI agent's reasoning process in real time, by reading periodic summaries of its internal reasoning as it works through a problem. Your job right now is purely to consolidate — not to judge or analyze, just to merge information into a clean running record.

    Below are two pieces of context:

    1. PRIOR CONTEXT (a condensed record of everything that happened before this point):
    {state['last_sum']}

    2. NEW ACTIVITY (the most recent summaries since the last check-in):
    {total_sum}

    Write an updated, single condensed narrative of the reasoning process so far, folding the new activity into the prior context. Requirements:
    - Roughly 200 words.
    - Chronological: earlier events first, most recent last.
    - Preserve concrete specifics that matter for tracking progress — what approach is being tried, what's been ruled out, what's been decided, any numbers/entities/results mentioned. Don't flatten everything into vague generalities.
    - If the new activity contradicts, revises, or abandons something from the prior context, reflect that change rather than just appending both versions.
    - Write it as a plain narrative paragraph, not a bulleted list.
    - Do not add commentary, opinions, or predictions about where this is headed — that happens in a later step. This is a factual consolidation only.

    Return only the narrative text, with no preamble or labels."""

    msg = await monitor_model.ainvoke(prompt)

    return {"curr_sum": msg.content}


async def analyze_node(state: MonitorState):

    prompt = f"""You are monitoring an AI agent's reasoning process. You've been given a condensed narrative of its progress so far. Your job is to assess *how the reasoning process itself is unfolding* — not to solve the underlying problem, and not to judge whether the agent's conclusions are correct.

    CONDENSED REASONING NARRATIVE:
    {state['curr_sum']}

    Write a roughly 100-word analysis addressing:
    - Is the reasoning moving toward a resolution, or circling without new progress?
    - Is it repeating the same approach/idea it already tried, or genuinely trying something new?
    - Are there signs of productive struggle (working through real difficulty) versus unproductive struggle (confusion, contradiction, or drift)?
    - Any concrete signal that it's converging on an answer soon, or conversely that it's diverging further from one?

    Base your analysis strictly on what's described in the narrative — do not speculate beyond it. Write it as plain prose, not a list. Return only the analysis text, with no preamble or labels."""

    msg = await monitor_model.ainvoke(prompt)
    return {"curr_analysis": msg.content}


async def classify_node(state: MonitorState):

    prompt = f"""You are the final classification step in a reasoning-monitor pipeline. Based on the analysis below, classify the current state of the AI agent's reasoning process into exactly one of these three categories:

    - Stuck: reasoning is repeating itself, contradicting earlier steps, or showing no meaningful forward movement over the recent activity.
    - Progressing: reasoning is actively advancing — trying new angles, narrowing possibilities, building on prior steps — even if it hasn't reached an answer yet.
    - Completed: the reasoning has reached a clear conclusion or final answer.

    ANALYSIS:
    {state['curr_analysis']}

    Respond with exactly one word: STUCK, PROGRESSING, or COMPLETED. No punctuation, no explanation, no additional text — the word alone, spelled and capitalized exactly as shown above."""

    msg = await monitor_model.ainvoke(prompt)
    type_enum = status[msg.content]
    return {"curr_state": type_enum}

def report_node(state:MonitorState):
    new_status_history = state['status_history']
    new_status_history.append(state['curr_state'])

    progress_handle.update(new_status_history)
    return {"status_history":new_status_history}

In [6]:
workflow = StateGraph(MonitorState)

workflow.add_node("compact_node", compact_node)
workflow.add_node("analyze_node", analyze_node)
workflow.add_node("classify_node", classify_node)
workflow.add_node("report_node", report_node)


workflow.add_edge(START, "compact_node")
workflow.add_edge("compact_node", "analyze_node")
workflow.add_edge("analyze_node", "classify_node")
workflow.add_edge("classify_node", "report_node")
workflow.add_edge("report_node", END)

chain = workflow.compile()

In [7]:
new_buffer = stream_buffer()
client = AsyncOpenAI()

prompt = """
Solve this logic puzzle. Five friends (Ann, Ben, Cara, Dan, Eve) each own a 
different pet (cat, dog, fish, bird, rabbit) and live on a different floor (1-5).

Clues:
1. Ann lives above the fish owner.
2. The dog owner lives on floor 1.
3. Cara owns the bird and lives on an odd floor.
4. Ben lives directly below Dan.
5. The rabbit owner lives on floor 4.
6. Eve does not live on floor 1 or floor 5.
7. Dan does not own the fish.
8. The cat owner lives above the dog owner but below the rabbit owner.
9. Ben owns the rabbit and lives on floor 2.

Do not give up or declare the puzzle unsolvable. Every clue is correct and a 
valid solution exists. Solve it, verify every clue against your solution one 
at a time, and if any clue fails, you have made an error — revise your solution 
and re-verify from clue 1. Repeat until you find the solution that satisfies 
every single clue simultaneously.
"""
stream = await client.responses.create(
    model="gpt-5.5",
    reasoning={"effort":"xhigh","summary":"auto"},
    input=[{"role":"user", "content":prompt}],
    stream=True,
)

In [ ]:
progress_handle = display("Starting...", display_id=True)

In [ ]:
compacted_sum = "No prior reasoning yet."
saved_state = status.PROGRESSING
prev_history = []


summaries = {}

prev_idx = None

async for event in stream:

    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

    elif event.type=="response.reasoning_summary_text.delta":
        print(event.delta, end="", flush=True)

        idx = event.summary_index
        if idx not in summaries:
            summaries[idx] = event.delta
        else:
            summaries[idx] += event.delta

        if prev_idx is not None and idx != prev_idx:
            new_buffer.append(summaries[prev_idx])
            print("Reasoning chunk appended")
            recent_sums = new_buffer.return_snapshot(1)
            result = await chain.ainvoke({"reasoning_sums":recent_sums, "last_sum":compacted_sum, "curr_sum":"", "curr_analysis":"", "curr_state":saved_state, "status_history":prev_history})
        
            compacted_sum = result["curr_sum"]
            saved_state = result["curr_state"]
            prev_history = result["status_history"]

        prev_idx = idx

# Invoke one last time to flush the last summary cycle - note that the state will be overwritten as complete 
if prev_idx is not None:
    new_buffer.append(summaries[prev_idx])
    final = await compact_node({
        "reasoning_sums": new_buffer.return_snapshot(1),
        "last_sum": compacted_sum,
    })
    compacted_sum = final["curr_sum"]

report_node({
    "reasoning_sums": [], "last_sum": "", "curr_sum": compacted_sum,
    "curr_analysis": "", "curr_state": status.COMPLETED,
    "status_history": prev_history,
})
            